# ⏰ Notebook 1: Why a Fixed Timeout Is Not Enough

Most failure detectors use a simple rule:

> *"If I haven't heard a heartbeat from you in `T` seconds, I'll mark you DEAD."*

Picking `T` is a **no-win situation**:

| If `T` is… | You get… |
|---|---|
| **too short** | false alarms every time the network hiccups (GC pause, packet loss, busy VM) |
| **too long** | real crashes take forever to notice — users stare at stale data |

In this notebook we **feel** that pain on a simulated noisy network.
After that, notebook 2 introduces the *phi accrual* detector, which replaces
the binary alive/dead flag with a smooth **suspicion score** that adapts to
the network's personality.

## Setup

```bash
cd 02-distributed-primitives/phi-accrual-failure-detection
uv sync
```

Then pick the `.venv` kernel (top-right of the notebook in VS Code). If it doesn't show up, `Cmd+Shift+P` → **Reload Window**.

## 1. Simulate a noisy heartbeat stream

We pretend a remote node sends a heartbeat every ~1 second, but the network adds ±0.4s of jitter (perfectly normal on real WANs). The node truly dies at `t = 20s`.

`heartbeats` is just a list of *arrival times* at our monitor.

In [ ]:
import random

random.seed(7)
INTERVAL = 1.0   # the node tries to beat every 1 second
JITTER   = 0.4   # ±0.4s of network noise
TOTAL    = 30.0  # total simulation time
DEAD_AT  = 20.0  # the node truly crashes at t=20s

now = 0.0
heartbeats = []
while now < TOTAL:
    now += INTERVAL + random.uniform(-JITTER, JITTER)
    if now >= DEAD_AT:
        break  # node is dead, no more beats
    heartbeats.append(now)

gaps = [b - a for a, b in zip(heartbeats, heartbeats[1:])]
print(f'received {len(heartbeats)} heartbeats before the crash at t={DEAD_AT}s')
print(f'inter-arrival gaps: min={min(gaps):.2f}s  max={max(gaps):.2f}s  mean={sum(gaps)/len(gaps):.2f}s')

# The jitter envelope is what makes timeout choice hard: some healthy gaps are well
# over 1.0s, so any timeout at or below max(gaps) WILL fire on a living node.
assert max(gaps) > 1.0, 'the trace should contain gaps longer than the nominal interval'
assert min(gaps) < 1.0
print(f'\nany fixed timeout <= {max(gaps):.2f}s fires at least once on a perfectly healthy node')

## 2. A fixed-timeout detector

Classic algorithm: every tick, ask *"how long since my last heartbeat?"* If that gap exceeds the timeout, declare the node DOWN. (New heartbeats bring it back UP.)

In [ ]:
def fixed_timeout_trace(beats, timeout, total=TOTAL, step=0.05):
    """Return (times, is_down_flags, first_down_time, false_alarms).

    * times: sampled timeline
    * is_down_flags: 1 if detector thinks node is DOWN at that time, else 0
    * first_down_time: when the detector first fired (None if never)
    * false_alarms: number of DOWN transitions BEFORE the real crash
    """
    times, flags = [], []
    last_seen = 0.0
    i = 0
    t = 0.0
    down = False
    first_down = None
    false_alarms = 0
    while t < total:
        # consume any heartbeats that have arrived by time t
        while i < len(beats) and beats[i] <= t:
            last_seen = beats[i]
            i += 1
            if down:  # recovered
                down = False
        if not down and (t - last_seen) > timeout:
            down = True
            if first_down is None:
                first_down = t
            if t < DEAD_AT:
                false_alarms += 1
        times.append(t)
        flags.append(1 if down else 0)
        t += step
    return times, flags, first_down, false_alarms

## 3. Try three different timeouts and visualize the tradeoff

Red bands = the detector *thinks* the node is dead. We want:

- **no** red bands before `t = 20s` (no false alarms while the node is alive),
- red to **start quickly** after `t = 20s` (fast detection once it really dies).

In [ ]:
import matplotlib.pyplot as plt

TIMEOUTS = [0.5, 1.5, 3.0]

fig, axes = plt.subplots(len(TIMEOUTS), 1, figsize=(10, 5), sharex=True)
for ax, to in zip(axes, TIMEOUTS):
    times, flags, first, fp = fixed_timeout_trace(heartbeats, to)
    ax.fill_between(times, 0, flags, step='pre', color='tab:red', alpha=0.3, label='detector says DOWN')
    ax.plot(heartbeats, [0.5]*len(heartbeats), '|', color='tab:green', markersize=14, label='heartbeat')
    ax.axvline(DEAD_AT, color='black', linestyle=':', label='real crash')
    ax.set_yticks([])
    delay = (first - DEAD_AT) if first is not None and first >= DEAD_AT else None
    title = f'timeout={to:.1f}s — false alarms before crash: {fp}'
    if delay is not None:
        title += f'   |   detected {delay:.1f}s after crash'
    ax.set_title(title, fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

# The trade-off, as numbers rather than as a picture: a short timeout has false
# alarms and a long one is slow, and there is no value that avoids both.
print(f"\n{'timeout':>8} {'false alarms':>13} {'detection delay':>17}")
table = {}
for to in (0.5, 1.0, 1.5, 3.0, 5.0):
    _, _, first, fp = fixed_timeout_trace(heartbeats, to)
    delay = (first - DEAD_AT) if first is not None and first >= DEAD_AT else None
    table[to] = (fp, delay)
    print(f'{to:>8.1f} {fp:>13} {(f"{delay:.2f}s" if delay is not None else "never"):>17}')

fps = [table[t][0] for t in sorted(table)]
delays = [table[t][1] for t in sorted(table) if table[t][1] is not None]
# False alarms fall as the timeout grows; detection delay rises. Monotone, opposed.
assert fps == sorted(fps, reverse=True), fps
assert delays == sorted(delays), delays
# And the crux: no single timeout gets both to their best value.
assert not any(fp == 0 and d is not None and d < 1.0 for fp, d in table.values()), table
print('\n✔ false alarms and detection delay move in opposite directions, and no timeout')
print('  in this range achieves zero false alarms AND sub-second detection')

## 4. What you should see

- **`timeout = 0.5s`** — way too jumpy. Every bit of jitter taller than 0.5s looks like a crash.
- **`timeout = 1.5s`** — reasonable, but you still might catch an unlucky gap, and you'll miss up to 1.5s after a real crash.
- **`timeout = 3.0s`** — no false alarms, but now real crashes take ~3s to notice.

The fundamental problem: **we picked one number for two very different questions** ("is this jitter?" vs. "is this a crash?"). And we assumed the network cadence is the same forever.

## 5. What we actually want

1. Learn the network's normal cadence automatically (mean + variance of inter-arrival times).
2. Output a **smooth suspicion score** that rises as silence stretches.
3. Let each caller pick its own threshold — a *cache* may reroute on mild suspicion, a *leader fencing* code path waits for near-certainty.

That is exactly what the **phi accrual failure detector** (Hayashibara et al., 2004) gives us, and it's why Cassandra, Akka, and ScyllaDB use it.

👉 Continue with `02_phi_accrual_detector.ipynb`.